# Tutorial: Power BI Measure Validation Showcase

This notebook is a guided demo for discovering a Power BI dataset, loading its measure validation template, executing selected validation cases, reviewing report context, and saving organized outputs under `test-results/`.


## Demo Flow

1. Authenticate with the repo's existing Power BI auth pattern.
2. List workspaces and choose one for the demo.
3. List datasets and choose the target semantic model.
4. Load the dataset's measure validation template or generated candidate file.
5. Filter and select validation cases.
6. Execute visible DAX queries and preview returned data.
7. Review related report metadata.
8. Save a clean result package for the run.


In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import Markdown, display

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'src').exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from scripts.helpers.measure_validation_demo import (
    build_execution_summary_frame,
    build_report_context,
    build_report_summary_markdown,
    execute_validation_case,
    filter_validation_cases,
    get_datasets_frame,
    get_reports_frame,
    get_workspaces_frame,
    load_validation_cases,
    locate_validation_template,
    pick_row,
    select_test_cases,
    validation_cases_to_rows,
)
from scripts.helpers.notebook_bootstrap import bootstrap_demo_notebook
from scripts.helpers.notebook_display import dataframe_preview, demo_error, markdown_summary, test_case_preview
from scripts.helpers.result_writer import (
    create_run_artifact_paths,
    write_json,
    write_markdown,
    write_queries,
    write_query_results,
    write_rows_to_csv,
)

context = bootstrap_demo_notebook(REPO_ROOT)
display(Markdown(markdown_summary('Notebook Context', {
    'Repo Root': context.repo_root,
    'Auth Mode': context.notebook_context.settings.auth_mode,
    'Config Environment': context.notebook_context.settings.environment_name,
})))


## 1. Configure the Demo

Edit the variables in the next cell if you want to steer the demo toward a known workspace, dataset, or subset of validation cases. If you leave them blank, the notebook falls back to the first available match.


In [ ]:
AUTH_MODE = None
USE_DEVICE_CODE = True

WORKSPACE_NAME_CONTAINS = ''
WORKSPACE_INDEX = 0
DATASET_NAME_CONTAINS = ''
DATASET_INDEX = 0

TEMPLATE_OVERRIDE = None
STATUS_FILTER = 'draft'
REVIEW_STATUS_FILTER = ''
PRIORITY_FILTER = 'high'
MEASURE_NAME_FILTER = ''
SCENARIO_TYPE_FILTER = ''
SELECTED_TEST_IDS = []
CASE_LIMIT = 5

RESULTS_BASE_DIR = REPO_ROOT / 'test-results'


## 2. Discover Workspaces and Datasets

This section authenticates with Power BI, lists visible workspaces, and picks a dataset for the rest of the notebook. The chosen workspace and dataset are displayed explicitly so the audience can follow the context.


In [ ]:
client = context.create_client(auth_mode=AUTH_MODE, use_device_code=USE_DEVICE_CODE)

workspace_frame = get_workspaces_frame(client)
display(Markdown(markdown_summary('Workspace Discovery', {
    'Workspace Count': len(workspace_frame),
    'Selection Hint': WORKSPACE_NAME_CONTAINS or '<first row>',
})))
display(dataframe_preview(workspace_frame, limit=10))

selected_workspace = pick_row(workspace_frame, preferred_name=WORKSPACE_NAME_CONTAINS, index=WORKSPACE_INDEX)
if selected_workspace is None:
    raise RuntimeError('No workspaces were returned for the current identity.')

workspace_id = selected_workspace['id']
dataset_frame = get_datasets_frame(client, workspace_id)
display(Markdown(markdown_summary('Selected Workspace', {
    'Workspace Name': selected_workspace.get('name', ''),
    'Workspace Id': workspace_id,
    'Dataset Count': len(dataset_frame),
})))
display(dataframe_preview(dataset_frame, limit=10))

selected_dataset = pick_row(dataset_frame, preferred_name=DATASET_NAME_CONTAINS, index=DATASET_INDEX)
if selected_dataset is None:
    raise RuntimeError('No datasets were returned for the selected workspace.')

dataset_id = selected_dataset['id']
dataset_name = selected_dataset.get('name', '')
reports_frame = get_reports_frame(client, workspace_id)

display(Markdown(markdown_summary('Selected Dataset', {
    'Dataset Name': dataset_name,
    'Dataset Id': dataset_id,
    'Configured By': selected_dataset.get('configuredBy', ''),
    'Reports In Workspace': len(reports_frame),
})))


## 3. Load the Measure Validation Template

The notebook first looks for a dataset-specific template. If that is missing, it falls back to the shared template or generated candidate file. The fallback is shown clearly so the audience can see whether the notebook is working from approved content or inferred draft content.


In [ ]:
template_selection = locate_validation_template(context.repo_root, dataset_name)
if TEMPLATE_OVERRIDE:
    template_selection = type(template_selection)(Path(TEMPLATE_OVERRIDE), 'manual_override', 'Using an explicit template override path.')

validation_frame = load_validation_cases(template_selection.path, dataset_name)
filtered_cases = filter_validation_cases(
    validation_frame,
    status=STATUS_FILTER,
    review_status=REVIEW_STATUS_FILTER,
    priority=PRIORITY_FILTER,
    measure_name=MEASURE_NAME_FILTER,
    scenario_type=SCENARIO_TYPE_FILTER,
)
selected_cases = select_test_cases(filtered_cases, selected_test_ids=SELECTED_TEST_IDS, limit=CASE_LIMIT)

display(Markdown(markdown_summary('Template Selection', {
    'Template Path': template_selection.path or '<missing>',
    'Template Source': template_selection.source,
    'Note': template_selection.note,
    'Available Cases': len(validation_frame),
    'Filtered Cases': len(filtered_cases),
    'Selected Cases': len(selected_cases),
})))

if validation_frame.empty:
    display(Markdown('**No validation cases were available for the selected dataset.**'))
else:
    display(test_case_preview(filtered_cases))


## 4. Execute Selected Test Cases

Each test case is executed individually. The notebook keeps the DAX query visible, shows a small result preview, and records a lightweight `pass` / `needs_review` / `error` outcome without pretending that machine inference replaces human review.


In [ ]:
execution_results = []

for test_case in validation_cases_to_rows(selected_cases):
    result = execute_validation_case(client, workspace_id, dataset_id, test_case)
    execution_results.append(result)

    display(Markdown(f"### {result['test_id']} :: {result['measure_name']}"))

    scenario_lines = "\n".join([
        f"- Scenario: `{result['scenario_type']}`",
        f"- Outcome: `{result['outcome']}`",
        f"- Expected Behavior: {result['expected_behavior']}",
    ])
    display(Markdown(scenario_lines))

    dax_block = "```sql\n" + result['dax_query'] + "\n```"
    display(Markdown(dax_block))

    if result['error']:
        display(Markdown(f"**Query Error**: `{result['error']}`"))
    elif result['rows']:
        display(pd.DataFrame(result['rows']).head(10))
    else:
        display(Markdown('_Query returned no rows._'))

execution_summary = build_execution_summary_frame(execution_results)
display(Markdown(markdown_summary('Execution Summary', {
    'Executed Cases': len(execution_results),
    'Pass Count': int((execution_summary['outcome'] == 'pass').sum()) if not execution_summary.empty else 0,
    'Needs Review Count': int((execution_summary['outcome'] == 'needs_review').sum()) if not execution_summary.empty else 0,
    'Error Count': int((execution_summary['outcome'] == 'error').sum()) if not execution_summary.empty else 0,
})))
display(execution_summary)


## 5. Review Report Context

This section shows workspace report metadata and, when the local PBIP demo report is available, measure-to-page-to-visual mappings for the selected measures. If report rendering is not practical, this metadata summary still gives clear report relevance.


In [ ]:
selected_measure_names = selected_cases['measure_name'].tolist() if not selected_cases.empty else []
report_context = build_report_context(context.repo_root, reports_frame, dataset_id, dataset_name, selected_measure_names)
report_summary_markdown = build_report_summary_markdown(report_context)

display(Markdown(report_summary_markdown))
if report_context['api_reports']:
    display(pd.DataFrame(report_context['api_reports']))
if report_context['local_measure_usage']:
    display(pd.DataFrame(report_context['local_measure_usage']))


## 6. Save a Demo-Friendly Result Package

The final section writes a clean timestamped run folder under `test-results/`. This keeps the notebook useful for demos and follow-up review without turning it into a larger application.


In [ ]:
artifact_paths = create_run_artifact_paths(RESULTS_BASE_DIR, dataset_name)
selected_case_rows = validation_cases_to_rows(selected_cases)
query_rows = [
    {
        'test_id': result['test_id'],
        'measure_name': result['measure_name'],
        'dax_query': result['dax_query'],
    }
    for result in execution_results
]
run_summary = {
    'workspace': selected_workspace,
    'dataset': selected_dataset,
    'template_path': str(template_selection.path) if template_selection.path else '',
    'template_source': template_selection.source,
    'selected_case_count': len(selected_case_rows),
    'executed_case_count': len(execution_results),
    'execution_outcomes': execution_summary.to_dict(orient='records') if not execution_summary.empty else [],
}

write_json(artifact_paths.run_summary_path, run_summary)
write_rows_to_csv(artifact_paths.selected_test_cases_path, selected_case_rows)
write_queries(artifact_paths.executed_queries_path, query_rows)
write_query_results(artifact_paths.query_results_dir, execution_results)
write_json(artifact_paths.report_metadata_path, report_context)
write_markdown(artifact_paths.report_summary_path, report_summary_markdown)

display(Markdown(markdown_summary('Artifacts Saved', {
    'Run Folder': artifact_paths.run_root,
    'Run Summary': artifact_paths.run_summary_path.name,
    'Selected Test Cases': artifact_paths.selected_test_cases_path.name,
    'Executed Queries': artifact_paths.executed_queries_path.name,
    'Query Result Files': len(execution_results),
    'Report Summary': artifact_paths.report_summary_path.name,
})))


## Next Step

If you want a more curated presentation, set `SELECTED_TEST_IDS` to a small list of reviewed test cases and rerun sections 3 through 6. That produces a cleaner narrative while preserving the same saved artifact structure.
